## 15.1 Setup:

In [47]:
# 15.1 Setup: retriever + corrective layer (with the gate fix), question sets with labels
import json
import re
import time
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv
from src.paths import CHUNKS
from src.retriever import Retriever
from src.query import SpellFixer
from src.corrective import Corrective
import src.answer as answer_mod
import src.evaluate as ev
from src.llm import chat, real_answer_check

ROOT = CHUNKS.parents[2]
EVAL = ROOT / "data" / "eval" / "benchmark"
EVIDENCE_FILE = EVAL / "prompt_ab_evidence.jsonl"
ANSWERS_FILE = EVAL / "prompt_ab_answers.csv"
load_dotenv(ROOT / ".env",override=True)

R = Retriever()
CR = Corrective(R, SpellFixer(R.chunks))
manual = pd.read_excel(EVAL / "manual_questions_with_answers.xlsx")
manual.columns = manual.columns.str.strip()
set_c = pd.read_csv(EVAL / "set_C_claude100.csv")
negatives = pd.read_csv(EVAL.parent / "negatives_50.csv")

QUESTIONS = pd.concat([
    pd.DataFrame({"source": "manual", "qid": manual.qid, "question": manual.question,
                  "label": "answerable", "expected": manual.expected_answer}),
    pd.DataFrame({"source": "set_C", "qid": set_c.qid, "question": set_c.question,
                  "label": set_c.expected.map({"answer": "answerable", "refuse": "unanswerable"}),
                  "expected": set_c.reference_answer.where(set_c.expected == "answer", "")}),
    pd.DataFrame({"source": "negatives", "qid": [f"N{index:02d}" for index in negatives.index],
                  "question": negatives.question, "label": "unanswerable", "expected": ""}),
], ignore_index=True)
display(pd.crosstab(QUESTIONS.source, QUESTIONS.label, margins=True))

label,answerable,unanswerable,All
source,,,
manual,23,0,23
negatives,0,50,50
set_C,52,48,100
All,75,98,173


## 15.2 Freeze the evidence (Algorithm 3, line 1)

In [48]:
# 15.2 Freeze the evidence (Algorithm 3, line 1): CR.prepare ONCE per question, saved to disk
def freeze(item):
    prep = CR.prepare(item.question)
    hits = [{key: hit[key] for key in ("chunk_id", "doc_type", "path", "printed_pages", "text")} for hit in prep["hits"]]
    return {"source": item.source, "qid": item.qid, "label": item.label, "question": item.question,
            "expected": item.expected, "fixed": prep["fixed"], "action": prep["action"], "final": prep["final"],
            "signals": prep["signals"], "hits": hits}

if EVIDENCE_FILE.exists():
    EVIDENCE = [json.loads(line) for line in EVIDENCE_FILE.open(encoding="utf-8")]
    
    print(f"loaded frozen evidence: {len(EVIDENCE)} questions (delete {EVIDENCE_FILE.name} to rebuild)")
else:
    EVIDENCE = [freeze(item) for item in QUESTIONS.itertuples()]
    EVIDENCE_FILE.write_text("\n".join(json.dumps(record, ensure_ascii=False) for record in EVIDENCE) + "\n", encoding="utf-8")
    print(f"froze evidence for {len(EVIDENCE)} questions → {EVIDENCE_FILE.name}")
display(pd.crosstab([record["source"] for record in EVIDENCE], [record["action"] for record in EVIDENCE], margins=True))

loaded frozen evidence: 173 questions (delete prompt_ab_evidence.jsonl to rebuild)


col_0,answer,answer_with_note,reject,scoped,All
row_0,,,,,
manual,19,1,3,0,23
negatives,0,9,25,16,50
set_C,56,19,23,2,100
All,75,29,51,18,173


##  15.2b

In [49]:
# 15.2b E0 - false premises the guideline can correct become answerable; the expected answer is the correction
PREMISE_QIDS = ["C-061", "C-063", "C-064", "C-065", "C-066", "C-067", "C-068"]
CORRECTIONS = {row.qid: re.sub(r"^(No|Wrong premise)\s*-\s*", "", row.reference_answer)
               for row in set_c[set_c.qid.isin(PREMISE_QIDS)].itertuples()}
for record in EVIDENCE:
    if record["qid"] in CORRECTIONS:
        record.update(label="answerable", expected=CORRECTIONS[record["qid"]])
print(f"{len(CORRECTIONS)} false premises relabelled as answerable:")
for qid, correction in CORRECTIONS.items():
    print(f"  {qid}: {correction}")

7 false premises relabelled as answerable:
  C-061: it is the Sexual Harassment of Women at Workplace (Prevention, Prohibition and Redressal) Act, 2013.
  C-063: PGMO doctors are recruited at a higher pay scale than MBBS doctors.
  C-064: contracts are reviewed periodically (say one year) and renewed if appropriate.
  C-065: league tables are suggested for facilities and specialties in DH/SDH/CHCs.
  C-066: the letter asks for personal monitoring every quarter.
  C-067: the grants are for male MPHW training centres set up prior to 2012.
  C-068: the MSG decisions were conveyed in the letter of 13th June 2016 (health systems approach to HR).


## 15.3 Prompt versions (system rules + weak-evidence note) and formats (Algorithm 3)

In [50]:
# # 15.3 Prompt versions (system rules + weak-evidence note) and formats (Algorithm 3)
# SYSTEM_V3 = f"""You answer questions about the "Guidelines on Human Resources for Health for NHM (2022)".
# How to answer:
# 1. Use all provided sources as needed. The answer may be in any source.
# 2. Use ONLY these sources. No outside knowledge, no guessing.
# 3. After each fact write the number of the source you read it in, e.g. [4].
# 4. Tables: read row by row and keep each value with its row and column names. For "only" questions, choose rows where that column is Yes and the other columns are No.
# 5. Length: for a single fact, answer in 1-3 sentences. When the question asks for a list, steps, a process, a form, a letter or "all" items, give every item from the sources as a bullet list.
# 6. If the sources answer only part of the question, answer that part and say which part the sources do not cover.
# 7. Only if none of the sources contains any answer to the question, reply exactly: {answer_mod.NOT_FOUND}"""

# def notes_v3(sig):
#     """Weak-evidence notes without the push to refuse: answer what the sources support, name what is missing."""
#     notes = []
#     if sig["soft_out"]:
#         notes.append(f"The sources do not mention {', '.join(sig['soft_out'])}. Say so, and answer only what the sources state about NHM.")
#     if sig["missing"]:
#         notes.append(f"The sources may not contain: {', '.join(sig['missing'])}. If so, say that this detail is not in the guideline.")
#     if sig["best"] < 0:
#         notes.append("The evidence may be weak: answer only what the sources clearly state.")
#     return notes

# VERSIONS = {"v1": (answer_mod.SYSTEM_PROMPT, CR.notes_for), "v3": (SYSTEM_V3, notes_v3)}

# def pages_of(hit):
#     return ", ".join(str(page) for page in hit["printed_pages"] if page is not None) or "?"

# def format_source_label(question, hits):
#     blocks = [f"Source [{n}]{' (table)' if hit['doc_type'] == 'table' else ''} - {hit['path']} - page {pages_of(hit)}:\n{hit['text']}"
#               for n, hit in enumerate(hits, 1)]
#     return "Sources:\n\n" + "\n\n".join(blocks) + f"\n\nQuestion: {question}\nAnswer:"

# def format_markdown(question, hits):
#     blocks = [f"### [{n}] {hit['path']} (page {pages_of(hit)}){' - table' if hit['doc_type'] == 'table' else ''}\n{hit['text']}"
#               for n, hit in enumerate(hits, 1)]
#     return f"## Question\n{question}\n\n## Sources\n" + "\n\n".join(blocks) + f"\n\n## Task\nAnswer the question: {question}"

# FORMATS = {"f0": answer_mod.build_prompt, "f1": format_source_label, "f2": format_markdown}

# def build(record, version, fmt):
#     system, notes_fn = VERSIONS[version]
#     notes = notes_fn(record["signals"]) if record["final"] == "P" else []
#     return system, ("Important: " + " ".join(notes) + "\n\n" if notes else "") + FORMATS[fmt](record["fixed"], record["hits"])

## 15.4 Generate answers (Algorithm 3, lines 2-6): resumable - every answer is saved at once, done ones are skipped

## run when API called is need, tested V1 and v3 prompt all coded down are comment so if kernal Fail we just don't re-run this cell

In [51]:
# # 15.4 Generate answers (Algorithm 3, lines 2-6): resumable - every answer is saved at once, done ones are skipped
# # MODEL = "groq:openai/gpt-oss-120b"
# MODEL = "gemini:gemini-3.8-flash"
# RUNS = 1 
# RUN_VERSIONS = ["v1", "v3"]
# RUN_FORMATS = ["f0"]          # stage 1: the current format only; add "f1", "f2" in stage 2
# RUNS = 2
# TO_ANSWER = [record for record in EVIDENCE if record["action"] in ("answer", "answer_with_note")]

# done = set()
# if ANSWERS_FILE.exists():
#     previous = pd.read_csv(ANSWERS_FILE)
#     done = set(zip(previous.model, previous.version, previous.format, previous.run, previous.qid))
# jobs = [(version, fmt, run, record) for version in RUN_VERSIONS for fmt in RUN_FORMATS for run in range(1, RUNS + 1)
#         for record in TO_ANSWER if (MODEL, version, fmt, run, record["qid"]) not in done]
# print(f"{len(TO_ANSWER)} questions reach the LLM · {len(jobs)} calls still to make ({len(done)} already saved)")
# print("model check:", real_answer_check(MODEL))

# start = time.time()
# for number, (version, fmt, run, record) in enumerate(jobs, 1):
#     system, prompt = build(record, version, fmt)
#     answer = chat(MODEL, system, prompt)
#     pd.DataFrame([{"model": MODEL, "version": version, "format": fmt, "run": run, "qid": record["qid"],
#                    "answer": answer}]).to_csv(ANSWERS_FILE, mode="a", header=not ANSWERS_FILE.exists(), index=False)
#     if number % 20 == 0:
#         print(f"  {number}/{len(jobs)} · {time.time() - start:.0f} s")
# print(f"done: {len(jobs)} answers in {time.time() - start:.0f} s → {ANSWERS_FILE.name}")

In [52]:
# # 15.4 Generate answers (Algorithm 3, lines 2-6): resumable - every answer is saved at once, done ones are skipped
# MODEL = "gemini:gemini-3.5-flash-lite"
# RUN_VERSIONS = ["v1", "v3"]
# RUN_FORMATS = ["f0"]          # stage 1: the current format only; add "f1", "f2" in stage 2
# RUNS = 1                      # start with 1 run per version; add the 2nd once we see the quota
# PAUSE = 5                     # seconds between calls, to stay under the free per-minute limit
# TO_ANSWER = [record for record in EVIDENCE if record["action"] in ("answer", "answer_with_note")]

# done = set()
# if ANSWERS_FILE.exists():
#     previous = pd.read_csv(ANSWERS_FILE)
#     done = set(zip(previous.model, previous.version, previous.format, previous.run, previous.qid))
# jobs = [(version, fmt, run, record) for version in RUN_VERSIONS for fmt in RUN_FORMATS for run in range(1, RUNS + 1)
#         for record in TO_ANSWER if (MODEL, version, fmt, run, record["qid"]) not in done]
# print(f"{len(TO_ANSWER)} questions reach the LLM · {len(jobs)} calls still to make ({len(done)} already saved)")
# print("model check:", real_answer_check(MODEL))

# start = time.time()
# for number, (version, fmt, run, record) in enumerate(jobs, 1):
#     system, prompt = build(record, version, fmt)
#     answer = chat(MODEL, system, prompt)
#     pd.DataFrame([{"model": MODEL, "version": version, "format": fmt, "run": run, "qid": record["qid"],
#                    "answer": answer}]).to_csv(ANSWERS_FILE, mode="a", header=not ANSWERS_FILE.exists(), index=False)
#     if number % 20 == 0:
#         print(f"  {number}/{len(jobs)} · {time.time() - start:.0f} s")
#     time.sleep(PAUSE)
# print(f"done: {len(jobs)} answers in {time.time() - start:.0f} s → {ANSWERS_FILE.name}")

In [53]:
# import src.llm as llm
# print(list(llm.API), llm.provider_of(MODEL) if "MODEL" in dir() else llm.provider_of("gemini:gemini-3.8-flash"))

In [54]:
# import importlib
# import src.llm as llm

# importlib.reload(llm)
# print("file loaded :", llm.__file__)
# print("has gemini  :", "gemini" in open(llm.__file__, encoding="utf-8").read())
# print("API after reload:", list(llm.API))

##  15.5 Score every run (Algorithms 1 + 2B) and decide (Algorithm 3, lines 7-11)

In [55]:
# # 15.5 Score every run (Algorithms 1 + 2B) and decide (Algorithm 3, lines 7-11)
# import importlib
# ev = importlib.reload(ev)
# MODEL = "gemini:gemini-3.5-flash-lite"
# BY_QID = {record["qid"]: record for record in EVIDENCE}
# answers = pd.read_csv(ANSWERS_FILE).fillna({"answer": ""})
# answers = answers[answers.qid.isin(BY_QID)]
# answers["source"] = answers.qid.map(lambda qid: BY_QID[qid]["source"])
# answers["label"] = answers.qid.map(lambda qid: BY_QID[qid]["label"])
# answers["question"] = answers.qid.map(lambda qid: BY_QID[qid]["question"])
# answers["expected"] = answers.qid.map(lambda qid: BY_QID[qid]["expected"] or "")
# answers["action"] = answers.qid.map(lambda qid: BY_QID[qid]["action"])
# answers["prompt"] = [build(BY_QID[row.qid], row.version, row.format)[1] for row in answers.itertuples()]
# answers["sources"] = answers.qid.map(lambda qid: [hit["text"] for hit in BY_QID[qid]["hits"]])
# answers = answers[answers.model == MODEL]
# SCORED = ev.score_run(answers)

# groups = SCORED.groupby(["model", "version", "format", "run"])
# RUN_TABLE = pd.DataFrame({key: {**ev.trust_score(group), **ev.rule_rates(group)} for key, group in groups}).T.astype(float)
# RUN_TABLE.index.names = ["model", "version", "format", "run"]
# display(RUN_TABLE)
# summary = RUN_TABLE.groupby(level=["model", "version"]).trust.agg(["min", "median", "max"])
# display(summary)
# display(pd.crosstab([SCORED.version, SCORED.source], SCORED.error.replace("", "ok")))

## 15.6 Decision (Algorithm 3, line 9) for one model: old = v1, new = v3

In [56]:
# # 15.6 Decision (Algorithm 3, line 9) for one model: old = v1, new = v3
# def decide(table, model, old="v1", new="v3"):
#     old_runs, new_runs = table.loc[(model, old)], table.loc[(model, new)]
#     noise = old_runs.trust.max() - old_runs.trust.min()
#     checks = {"median gain beats noise": new_runs.trust.median() - old_runs.trust.median() > noise,
#               "over-answer not up": new_runs.over_answer.mean() <= old_runs.over_answer.mean(),
#               "worst case not worse": new_runs.trust.min() >= old_runs.trust.min()}
#     print(f"{model}: noise {noise:.3f} · median {old} {old_runs.trust.median():.3f} → {new} {new_runs.trust.median():.3f}")
#     for name, passed in checks.items():
#         print(f"  {'PASS' if passed else 'FAIL'}  {name}")
#     print("  →", "ADOPT " + new if all(checks.values()) else "KEEP " + old)

# decide(RUN_TABLE, MODEL)

In [57]:
# # 15.7 Inspect v3's over-answers next to v1's answer to the same question
# RUNS_OF_MODEL = SCORED[SCORED.model == MODEL]
# over = RUNS_OF_MODEL[(RUNS_OF_MODEL.version == "v3") & (RUNS_OF_MODEL.error == "over-responsiveness")]
# v1_answers = RUNS_OF_MODEL[RUNS_OF_MODEL.version == "v1"].set_index("qid").answer

# for row in over.sort_values(["source", "qid"]).itertuples():
#     record = BY_QID[row.qid]
#     notes = notes_v3(record["signals"]) if record["final"] == "P" else []
#     print(f"{row.qid} | {row.source} | label={row.label} | action={row.action} | final={record['final']}")
#     print("Q:", row.question)
#     print("expected:", (row.expected or "-")[:200])
#     print("notes given:", notes or "-")
#     print("v3:", row.answer.strip()[:600])
#     print("v1:", v1_answers.get(row.qid, "").strip()[:300])
#     print("-" * 100)

In [58]:
# # 15.7 Inspect v3's over-answers next to v1's answer to the same question
# RUNS_OF_MODEL = SCORED[SCORED.model == MODEL]
# over = RUNS_OF_MODEL[(RUNS_OF_MODEL.version == "v3") & (RUNS_OF_MODEL.error == "improper citation")]
# v1_answers = RUNS_OF_MODEL[RUNS_OF_MODEL.version == "v1"].set_index("qid").answer

# for row in over.sort_values(["source", "qid"]).itertuples():
#     record = BY_QID[row.qid]
#     notes = notes_v3(record["signals"]) if record["final"] == "P" else []
#     print(MODEL)
#     print(f"{row.qid} | {row.source} | label={row.label} | action={row.action} | final={record['final']}")
#     print("Q:", row.question)
#     print("expected:", (row.expected or "-")[:200])
#     print("notes given:", notes or "-")
#     print("v3:", row.answer.strip()[:600])
#     print("v1:", v1_answers.get(row.qid, "").strip()[:300])
#     print("-" * 100)


In [59]:
# # 15.4 Generate answers (Algorithm 3, lines 2-6): resumable - every answer is saved at once, done ones are skipped
# from src.llm import LLMError

# MODEL = "groq:openai/gpt-oss-120b"
# RUN_VERSIONS = ["v1", "v3"]
# RUN_FORMATS = ["f0"]          # stage 1: the current format only; add "f1", "f2" in stage 2
# RUNS = 2
# PAUSE = 30                    # seconds between calls: Groq allows 8K tokens/minute, one answer uses ~3-5K
# TO_ANSWER = [record for record in EVIDENCE if record["action"] in ("answer", "answer_with_note")]

# done = set()
# if ANSWERS_FILE.exists():
#     previous = pd.read_csv(ANSWERS_FILE)
#     done = set(zip(previous.model, previous.version, previous.format, previous.run, previous.qid))
# jobs = [(version, fmt, run, record) for run in range(1, RUNS + 1) for version in RUN_VERSIONS for fmt in RUN_FORMATS
#         for record in TO_ANSWER if (MODEL, version, fmt, run, record["qid"]) not in done]
# print(f"{len(TO_ANSWER)} questions reach the LLM · {len(jobs)} calls still to make ({len(done)} already saved)")
# print("model check:", real_answer_check(MODEL))

# start = time.time()
# for number, (version, fmt, run, record) in enumerate(jobs, 1):
#     system, prompt = build(record, version, fmt)
#     try:
#         answer = chat(MODEL, system, prompt)
#     except LLMError as error:
#         print(f"stopped after {number - 1}/{len(jobs)} answers: {error}\nrerun this cell after the limit resets")
#         break
#     pd.DataFrame([{"model": MODEL, "version": version, "format": fmt, "run": run, "qid": record["qid"],
#                    "answer": answer}]).to_csv(ANSWERS_FILE, mode="a", header=not ANSWERS_FILE.exists(), index=False)
#     if number % 20 == 0:
#         print(f"  {number}/{len(jobs)} · {time.time() - start:.0f} s")
#     time.sleep(PAUSE)
# else:
#     print(f"done: {len(jobs)} answers in {time.time() - start:.0f} s → {ANSWERS_FILE.name}")

In [60]:
# # 15.5 Score every run (Algorithms 1 + 2B) and decide (Algorithm 3, lines 7-11)
# import importlib
# ev = importlib.reload(ev)
# BY_QID = {record["qid"]: record for record in EVIDENCE}
# answers = pd.read_csv(ANSWERS_FILE).fillna({"answer": ""})
# answers = answers[answers.qid.isin(BY_QID)]
# answers["source"] = answers.qid.map(lambda qid: BY_QID[qid]["source"])
# answers["label"] = answers.qid.map(lambda qid: BY_QID[qid]["label"])
# answers["question"] = answers.qid.map(lambda qid: BY_QID[qid]["question"])
# answers["expected"] = answers.qid.map(lambda qid: BY_QID[qid]["expected"] or "")
# answers["action"] = answers.qid.map(lambda qid: BY_QID[qid]["action"])
# answers["prompt"] = [build(BY_QID[row.qid], row.version, row.format)[1] for row in answers.itertuples()]
# answers["sources"] = answers.qid.map(lambda qid: [hit["text"] for hit in BY_QID[qid]["hits"]])
# answers = answers[answers.model == MODEL]
# SCORED = ev.score_run(answers)

# groups = SCORED.groupby(["model", "version", "format", "run"])
# RUN_TABLE = pd.DataFrame({key: {**ev.trust_score(group), **ev.rule_rates(group)} for key, group in groups}).T.astype(float)
# RUN_TABLE.index.names = ["model", "version", "format", "run"]
# display(RUN_TABLE)
# summary = RUN_TABLE.groupby(level=["model", "version"]).trust.agg(["min", "median", "max"])
# display(summary)
# # display(pd.crosstab([SCORED.version, SCORED.source], SCORED.error.replace("", "ok")))
# display(pd.crosstab([SCORED.model, SCORED.version, SCORED.source], SCORED.error.replace("", "ok")))

In [61]:
# decide(RUN_TABLE, "groq:openai/gpt-oss-120b")

In [62]:
# # 15.8 Citation check: which citation styles each model writes, and what the scorer sees in "improper citation" answers
# STYLES = {"[n]": r"\[\d+\]", "[n, m]": r"\[\d+(?:\s*[,;–-]\s*\d+)+\]", "【n】": r"【[^】]*】",
#           "Source n": r"\b[Ss]ource\s+\d+", "table rows": r"(?m)^\s*\|"}
# style_table = pd.DataFrame({name: SCORED.answer.str.contains(pattern, regex=True) for name, pattern in STYLES.items()})
# display(style_table.groupby([SCORED.model, SCORED.version]).mean().round(2))

# GROQ = "groq:openai/gpt-oss-120b"
# bad = SCORED[(SCORED.model == GROQ) & (SCORED.version == "v1") & (SCORED.run == 1) & (SCORED.error == "improper citation")]
# for row in bad.head(6).itertuples():
#     print(f"{row.qid} | cite_recall={row.cite_recall} | statements={row.statements}")
#     print(row.answer.strip()[:700])
#     print("scorer sees:", [(text[:60], cites) for text, cites in ev.statements(row.answer)][:6])
#     print("-" * 100)

In [63]:
# # check_groq.py - can Groq take a REAL-sized request now? (asks for 2048 tokens like 15.4 does)
# import os
# import requests
# from dotenv import load_dotenv

# load_dotenv(r"D:\SwasthyaSetu\.env")
# reply = requests.post(
#     "https://api.groq.com/openai/v1/chat/completions",
#     headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"},
#     json={"model": "openai/gpt-oss-120b", "messages": [{"role": "user", "content": "Reply OK"}], "max_tokens": 2048},
#     timeout=60,
# )
# print("status:", reply.status_code)
# print("ready - run 15.4" if reply.ok else reply.text)

In [64]:
# import src.llm as llm
# print(llm.__file__, list(llm.API))

In [65]:
# from pathlib import Path
# import src.llm as llm
# path = Path(llm.__file__)
# print(path)
# print("nvidia in file on disk:", "nvidia" in path.read_text(encoding="utf-8"))
# print("nvidia in loaded module:", "nvidia" in llm.API)

In [66]:
# import importlib
# import src.llm as llm
# llm = importlib.reload(llm)
# from src.llm import API, LLMError, chat, real_answer_check, provider_of
# print(list(API))

In [67]:
# 15.9 Which models can each key call? Reads each provider's model list - no answers, no tokens used
import os
import requests
from src.llm import API

load_dotenv(ROOT / ".env")                    # picks up the new NVIDIA / OpenRouter keys
MODEL_LISTS = {"groq": "https://api.groq.com/openai/v1/models",
               "nvidia": "https://integrate.api.nvidia.com/v1/models",
               "openrouter": "https://openrouter.ai/api/v1/models"}

def model_ids(provider):
    key = os.environ.get(API[provider][1], "")
    reply = requests.get(MODEL_LISTS[provider], headers={"Authorization": f"Bearer {key}"}, timeout=60)
    reply.raise_for_status()
    return sorted(model["id"] for model in reply.json()["data"])

for provider in MODEL_LISTS:
    ids = model_ids(provider)
    if provider == "openrouter":
        ids = [model_id for model_id in ids if model_id.endswith(":free")]     # free models only
    print(f"{provider} ({len(ids)} models):\n  " + "\n  ".join(ids) + "\n")

groq (11 models):
  allam-2-7b
  canopylabs/orpheus-arabic-saudi
  canopylabs/orpheus-v1-english
  meta-llama/llama-prompt-guard-2-22m
  meta-llama/llama-prompt-guard-2-86m
  openai/gpt-oss-120b
  openai/gpt-oss-20b
  openai/gpt-oss-safeguard-20b
  qwen/qwen3.8-27b
  whisper-large-v3
  whisper-large-v3-turbo

nvidia (81 models):
  01-ai/yi-large
  adept/fuyu-8b
  ai21labs/jamba-1.5-large-instruct
  aisingapore/sea-lion-7b-instruct
  bigcode/starcoder2-15b
  databricks/dbrx-instruct
  deepseek-ai/deepseek-coder-6.7b-instruct
  deepseek-ai/deepseek-v4.1-flash
  google/codegemma-1.1-7b
  google/codegemma-7b
  google/deplot
  google/diffusiongemma-26b-a4b-it
  google/gemma-2b
  google/gemma-3-12b-it
  google/gemma-3-4b-it
  google/gemma-4-31b-it
  google/recurrentgemma-2b
  ibm/granite-3.0-3b-a800m-instruct
  ibm/granite-3.0-8b-instruct
  ibm/granite-34b-code-instruct
  ibm/granite-8b-code-instruct
  meta/codellama-70b
  meta/llama-3.2-11b-vision-instruct
  meta/llama-3.2-90b-vision-instru

In [68]:
# 15.10 Screening set: a fixed, stratified 40 of the 104 questions that reach the LLM (same frozen evidence)
SCREEN_FILE = EVAL / "screen_40.csv"
QUOTA = {"manual": 10, "set_C": 25, "negatives": 5}
POOL = pd.DataFrame([{key: record[key] for key in ("qid", "source", "label")}
                     for record in EVIDENCE if record["action"] in ("answer", "answer_with_note")])

def stratified(source, size):
    """`size` questions of one source (or all it has), keeping its answerable / unanswerable mix."""
    group = POOL[POOL.source == source]
    parts = [group[group.label == label] for label in group.label.unique()]
    return pd.concat([part.sample(min(len(part), round(size * len(part) / len(group))), random_state=7) for part in parts])

if SCREEN_FILE.exists():
    SCREEN = pd.read_csv(SCREEN_FILE)
    print(f"loaded {len(SCREEN)} screening questions (delete {SCREEN_FILE.name} to draw again)")
else:
    SCREEN = pd.concat([stratified(source, size) for source, size in QUOTA.items()], ignore_index=True)
    SCREEN.to_csv(SCREEN_FILE, index=False)
    print(f"saved {len(SCREEN)} screening questions → {SCREEN_FILE.name}")
display(pd.crosstab(SCREEN.source, SCREEN.label, margins=True))

loaded 40 screening questions (delete screen_40.csv to draw again)


label,answerable,unanswerable,All
source,,,
manual,10,0,10
negatives,0,5,5
set_C,17,8,25
All,27,13,40


## run when API call is need it takes 2 hour to run

In [69]:
# # 15.11 Screen many models on the 40 questions: prompt v1, format f0, 2 runs - resumable, one model at a time
# from src.llm import LLMError, provider_of
# SCREEN_MODELS = [
#     "gemini:gemini-3.5-flash-lite",
#     "groq:openai/gpt-oss-120b",
#     "groq:openai/gpt-oss-20b",
#     "groq:qwen/qwen3.8-27b",
#     "nvidia:deepseek-ai/deepseek-v4.1-flash",
#     "nvidia:google/gemma-4-31b-it",
#     "nvidia:mistralai/mistral-large-2-instruct",
#     "nvidia:moonshotai/kimi-k2.6",
#     "nvidia:nvidia/nemotron-3-super-120b-a12b",
#     "nvidia:nvidia/llama-3.1-nemotron-70b-instruct",
#     "nvidia:z-ai/glm-5.3-flash",
#     "nvidia:mistralai/mistral-7b-instruct-v0.3",
#     "nvidia:writer/palmyra-med-70b",
#     "openrouter:thinkingmachines/inkling:free",
# ]
# SCREEN_RUNS = 2
# PAUSE = {"groq": 20, "gemini": 5, "nvidia": 3, "openrouter": 4, "ollama": 0}   # seconds between calls per free tier
# BY_QID = {record["qid"]: record for record in EVIDENCE}
# SCREEN_RECORDS = [BY_QID[qid] for qid in SCREEN.qid]

# def saved_keys():
#     if not ANSWERS_FILE.exists():
#         return set()
#     previous = pd.read_csv(ANSWERS_FILE)
#     return set(zip(previous.model, previous.version, previous.format, previous.run, previous.qid))

# def screen_model(model):
#     """Answers the screening questions with one model; a limit or error stops this model only, not the others."""
#     done = saved_keys()
#     jobs = [(run, record) for run in range(1, SCREEN_RUNS + 1) for record in SCREEN_RECORDS
#             if (model, "v1", "f0", run, record["qid"]) not in done]
#     if not jobs:
#         return "complete"
#     try:
#         real_answer_check(model)
#     except LLMError as error:
#         return f"skipped - {str(error)[:200]}"
#     pause = PAUSE[provider_of(model)[0]]
#     for number, (run, record) in enumerate(jobs, 1):
#         system, prompt = build(record, "v1", "f0")
#         try:
#             answer = chat(model, system, prompt)
#         except LLMError as error:
#             return f"stopped after {number - 1}/{len(jobs)} - {str(error)[:200]}"
#         pd.DataFrame([{"model": model, "version": "v1", "format": "f0", "run": run, "qid": record["qid"],
#                        "answer": answer}]).to_csv(ANSWERS_FILE, mode="a", header=not ANSWERS_FILE.exists(), index=False)
#         time.sleep(pause)
#     return f"done - {len(jobs)} answers"

# for model in SCREEN_MODELS:
#     start = time.time()
#     print(f"{model}: {screen_model(model)} ({time.time() - start:.0f} s)", flush=True)

In [70]:
# # 15.12 Score the screening: one row per model (v1, f0, the 40 questions) - mean of the runs and their spread
# def attach_evidence(answers):
#     """Adds what the scorer needs: question data, the exact prompt that was sent and the source texts."""
#     answers = answers.copy()
#     for column in ("source", "label", "question", "action"):
#         answers[column] = answers.qid.map(lambda qid: BY_QID[qid][column])
#     answers["expected"] = answers.qid.map(lambda qid: BY_QID[qid]["expected"] or "")
#     answers["prompt"] = [build(BY_QID[row.qid], row.version, row.format)[1] for row in answers.itertuples()]
#     answers["sources"] = answers.qid.map(lambda qid: [hit["text"] for hit in BY_QID[qid]["hits"]])
#     return answers

# screen_answers = pd.read_csv(ANSWERS_FILE).fillna({"answer": ""})
# screen_answers = screen_answers[screen_answers.qid.isin(set(SCREEN.qid)) & screen_answers.model.isin(SCREEN_MODELS)
#                                 & screen_answers.version.eq("v1") & screen_answers.format.eq("f0")
#                                 & screen_answers.run.le(SCREEN_RUNS)]
# SCREEN_SCORED = ev.score_run(attach_evidence(screen_answers))

# per_run = pd.DataFrame({key: {**ev.trust_score(group), **ev.rule_rates(group)}
#                         for key, group in SCREEN_SCORED.groupby(["model", "run"])}).T.astype(float)
# per_run.index.names = ["model", "run"]
# by_model = per_run.groupby(level="model")
# BOARD = by_model.mean().round(3)
# BOARD["noise"] = by_model.trust.agg(lambda values: values.max() - values.min()).round(3)
# BOARD["runs"] = by_model.size()
# BOARD["empty answers"] = SCREEN_SCORED.groupby("model").answer.agg(lambda texts: texts.str.strip().eq("").sum())
# display(BOARD.sort_values("trust", ascending=False))
# display(pd.crosstab(SCREEN_SCORED.model, SCREEN_SCORED.error.replace("", "ok")))
# STYLES = {"[n]": r"\[\d+\]", "[n, m]": r"\[\d+(?:\s*[,;–-]\s*\d+)+\]", "【n】": r"【[^】]*】",
#           "Source n": r"\b[Ss]ource\s+\d+", "table rows": r"(?m)^\s*\|"}
# display(pd.DataFrame({name: SCREEN_SCORED.answer.str.contains(pattern, regex=True) for name, pattern in STYLES.items()})
#         .groupby(SCREEN_SCORED.model).mean().round(2))

In [71]:
# display(SCREEN_SCORED.groupby(["model", "run"]).size().unstack())

In [72]:
# # 15.13 Step 1a - set C "refuse" questions: which are false premises the guideline can correct?
# refuse = set_c[set_c.expected == "refuse"].copy()
# refuse["reaches_llm"] = refuse.qid.isin(set(POOL.qid))
# print("set C columns:", list(set_c.columns))
# for row in refuse[refuse.reaches_llm].itertuples():
#     print(f"{row.qid} | {row.question}")
#     print("   reference:", str(row.reference_answer)[:300])
# print(f"\n{refuse.reaches_llm.sum()} of {len(refuse)} refuse questions reach the LLM")

In [73]:
# BOARD = BOARD[(BOARD.runs == SCREEN_RUNS) & (BOARD.n == len(SCREEN))]     # only models that answered every question in every run
# BOARD

In [74]:
# # 15.15 Scorecard - only the three numbers our goal depends on (complete models only)
# def scorecard(scored):
#     refused = scored.refused.astype(bool)
#     answerable, unanswerable = scored.llm_label.eq("answerable"), scored.llm_label.eq("unanswerable")
#     correct = ~refused & scored.correctness.astype(float).fillna(0).ge(ev.TAU)
#     cited = scored.cite_recall.astype(float)[~refused].dropna()
#     return pd.Series({"correct on answerable %": 100 * correct[answerable].mean(),
#                       "over-answer on unanswerable %": 100 * (~refused)[unanswerable].mean(),
#                       "statements cited %": 100 * cited.mean(),
#                       "answers": len(scored)})

# CARD = SCREEN_SCORED.groupby("model").apply(scorecard).round(1)
# display(CARD[CARD.answers == SCREEN_RUNS * len(SCREEN)].sort_values("correct on answerable %", ascending=False))

In [75]:
# # 15.16 E1 input - which unanswerable questions are still over-answered, by how many models, and of what kind
# SWAPPED = {f"C-0{number}" for number in range(53, 61)}          # C-053 ... C-060: fact exists for another state/year

# def kind_of(qid, source, label):
#     if source == "negatives":
#         return "out of scope"            # → E1 gate can block in code
#     if label == "answerable":
#         return "retrieval miss"          # answerable, but its facts were not in the prompt
#     return "swapped entity" if qid in SWAPPED else "not covered"   # → E2 prompt rule

# over = SCREEN_SCORED[SCREEN_SCORED.llm_label.eq("unanswerable") & ~SCREEN_SCORED.refused.astype(bool)]
# table = (over.groupby(["qid", "source", "label"])
#              .agg(models=("model", "nunique"), answers=("answer", "size"),
#                   question=("question", "first"), example=("answer", "first"))
#              .reset_index())
# table["kind"] = [kind_of(qid, source, label) for qid, source, label in zip(table.qid, table.source, table.label)]
# table["example"] = table.example.str[:150]
# display(table.sort_values("answers", ascending=False))
# display(table.groupby("kind").answers.sum())

## test v1 vs v3 mediumm & v3 strict

In [76]:
# 15.3b v3b prompts: two strictness levels; formatting rules identical, only the "decide first" rules differ
FORMAT_RULES = """When you answer:
- Use ONLY these sources. No outside knowledge, no guessing.
- After each fact, and after each bullet, write the number of the source you read it in, e.g. [4].
- Tables: read row by row and keep each value with its row and column names. For "only" questions, choose rows where that column is Yes and the other columns are No.
- Length: for a single fact, answer in 1-3 sentences. When the question asks for a list, steps, a process, a form, a letter or "all" items, give every item from the sources as a bullet list."""

SYSTEM_V3B_MED = f"""You answer questions about the "Guidelines on Human Resources for Health for NHM (2022)".
First decide how to answer. Check these in order and use the FIRST that applies:
1. The question is about an organisation, place or topic the sources do not mention: reply only "The guideline does not cover <it>." Do not add NHM rules.
2. The question states something as true that the sources contradict: correct it with the value the sources state.
3. The question asks about one state, year, cadre or letter, and the sources give it only for a different one: reply only "The guideline does not cover this for <the one asked>." Do not give the other one's value.
4. The sources answer only part of the question: answer that part, then say which part the guideline does not cover.
5. None of the sources contains any answer: reply exactly: {answer_mod.NOT_FOUND}
6. Otherwise answer the question.
{FORMAT_RULES}"""

SYSTEM_V3B_STRICT = f"""You answer questions about the "Guidelines on Human Resources for Health for NHM (2022)".
First decide how to answer. Check these in order and use the FIRST that applies:
1. The question is about an organisation, place or topic the sources do not mention: reply exactly: {answer_mod.NOT_FOUND}
2. The question states something as true that the sources contradict: correct it, but only if a source states the correct value directly; otherwise reply exactly: {answer_mod.NOT_FOUND}
3. The question asks about one state, year, cadre or letter, and the sources give it only for a different one: reply exactly: {answer_mod.NOT_FOUND}
4. The sources do not state the answer to EVERY part of the question: reply exactly: {answer_mod.NOT_FOUND}
5. Otherwise answer the question.
{FORMAT_RULES}"""

def notes_v3b_med(sig):
    """Weak-evidence notes that agree with SYSTEM_V3B_MED (refuse the out-of-scope part, never fill it with NHM rules)."""
    notes = []
    if sig["soft_out"]:
        names = ", ".join(sig["soft_out"])
        notes.append(f"The sources do not mention {names}. If the question is about {names}, reply only: The guideline does not cover {names}.")
    if sig["missing"]:
        notes.append(f"The sources may not contain: {', '.join(sig['missing'])}. If they do not, say the guideline does not cover it; do not use a value given for something else.")
    if sig["best"] < 0:
        notes.append("The evidence may be weak: answer only what the sources clearly state.")
    return notes

def notes_v3b_strict(sig):
    """Weak-evidence notes that agree with SYSTEM_V3B_STRICT (any doubt -> the exact refusal)."""
    notes = []
    if sig["soft_out"]:
        names = ", ".join(sig["soft_out"])
        notes.append(f"The sources do not mention {names}. If the question is about {names}, reply exactly: {answer_mod.NOT_FOUND}")
    if sig["missing"]:
        notes.append(f"The sources may not contain: {', '.join(sig['missing'])}. If they do not, reply exactly: {answer_mod.NOT_FOUND}")
    if sig["best"] < 0:
        notes.append(f"The evidence may be weak: if the sources do not clearly state the answer, reply exactly: {answer_mod.NOT_FOUND}")
    return notes
VERSIONS = {"v1": (answer_mod.SYSTEM_PROMPT, CR.notes_for)}
VERSIONS.update({"v3b_med": (SYSTEM_V3B_MED, notes_v3b_med), "v3b_strict": (SYSTEM_V3B_STRICT, notes_v3b_strict)})
print(list(VERSIONS))

def format_source_label(question, hits):
    blocks = [f"Source [{n}]{' (table)' if hit['doc_type'] == 'table' else ''} - {hit['path']} - page {pages_of(hit)}:\n{hit['text']}"
              for n, hit in enumerate(hits, 1)]
    return "Sources:\n\n" + "\n\n".join(blocks) + f"\n\nQuestion: {question}\nAnswer:"

def pages_of(hit):
    return ", ".join(str(page) for page in hit["printed_pages"] if page is not None) or "?"

def format_markdown(question, hits):
    blocks = [f"### [{n}] {hit['path']} (page {pages_of(hit)}){' - table' if hit['doc_type'] == 'table' else ''}\n{hit['text']}"
              for n, hit in enumerate(hits, 1)]
    return f"## Question\n{question}\n\n## Sources\n" + "\n\n".join(blocks) + f"\n\n## Task\nAnswer the question: {question}"
FORMATS = {"f0": answer_mod.build_prompt, "f1": format_source_label, "f2": format_markdown}

def build(record, version, fmt):
    system, notes_fn = VERSIONS[version]
    notes = notes_fn(record["signals"]) if record["final"] == "P" else []
    return system, ("Important: " + " ".join(notes) + "\n\n" if notes else "") + FORMATS[fmt](record["fixed"], record["hits"])

['v1', 'v3b_med', 'v3b_strict']


In [77]:
# C-003 (Phase 2, step 1a): save E2's scoring input once, with the exact prompt of every answer (no API calls)
import hashlib, subprocess

SCORING_INPUT = EVAL / "e2_scoring_input.jsonl"
BY_QID = {record["qid"]: record for record in EVIDENCE}

def git(*args):
    return subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True, check=True).stdout.strip()

answers = pd.read_csv(ANSWERS_FILE).fillna({"answer": ""})
known = answers.version.isin(VERSIONS) & answers.format.isin(FORMATS) & answers.qid.isin(BY_QID)
print("left out (prompt version no longer defined):", answers[~known].groupby(["model", "version"]).size().to_dict())

rows = answers[known].reset_index(drop=True)
for column in ("source", "label", "question", "action"):
    rows[column] = rows.qid.map(lambda qid: BY_QID[qid][column])
rows["expected"] = rows.qid.map(lambda qid: BY_QID[qid]["expected"] or "")
built = [build(BY_QID[row.qid], row.version, row.format) for row in rows.itertuples()]
rows["system"] = [system for system, prompt in built]
rows["prompt"] = [prompt for system, prompt in built]
rows["prompt_hash"] = [hashlib.sha256(f"{system}\n\n{prompt}".encode("utf-8")).hexdigest()[:12] for system, prompt in built]
rows["sources"] = rows.qid.map(lambda qid: [hit["text"] for hit in BY_QID[qid]["hits"]])
rows["prompt_origin"] = "reconstructed"
rows["code_commit"] = git("rev-parse", "--short", "HEAD")
rows.to_json(SCORING_INPUT, orient="records", lines=True, force_ascii=False)

saved = pd.read_json(SCORING_INPUT, lines=True, dtype=False)
scoring_columns = ["model", "version", "format", "run", "qid", "answer", "source", "label",
                   "question", "action", "expected", "prompt", "sources"]
print("uncommitted changes in src/:", git("status", "--short", "src") or "none")
print("rows saved:", len(saved), "| empty prompts:", saved.prompt.str.strip().eq("").sum())
print(saved.groupby(["model", "version"]).size().to_string())
print("scores identical after save + reload:",
      ev.score_run(saved[scoring_columns]).equals(ev.score_run(rows[scoring_columns])))

left out (prompt version no longer defined): {('gemini:gemini-3.5-flash-lite', 'v3'): 104, ('groq:openai/gpt-oss-120b', 'v3'): 208}
uncommitted changes in src/: M src/evaluate.py
rows saved: 1297 | empty prompts: 0
model                                     version   
gemini:gemini-3.5-flash-lite              v1            144
                                          v3b_med        80
                                          v3b_strict     80
gemini:gemini-3.8-flash                   v1             19
groq:openai/gpt-oss-120b                  v1            208
groq:openai/gpt-oss-20b                   v1             80
groq:qwen/qwen3.8-27b                     v1             80
                                          v3b_med        80
                                          v3b_strict     80
nvidia:deepseek-ai/deepseek-v4.1-flash    v1             46
nvidia:google/gemma-4-31b-it              v1             80
nvidia:nvidia/nemotron-3-super-120b-a12b  v1             80
nvidia:z-ai/

In [78]:
# import inspect
# import src.llm as llm
# print(inspect.signature(llm.real_answer_check))
# print("in file on disk:", "def real_answer_check(model: str, timeout" in open(llm.__file__, encoding="utf-8").read())

In [79]:
# # 15.11 Screen models x prompt versions on the 40 questions (format f0, 2 runs) - resumable, one pair at a time
# from src.llm import LLMError, provider_of

# SCREEN_MODELS = [
#     "gemini:gemini-3.5-flash-lite",
#     "groq:openai/gpt-oss-120b",
#     "groq:openai/gpt-oss-20b",
#     "groq:qwen/qwen3.8-27b",
#     "nvidia:google/gemma-4-31b-it",
#     "nvidia:nvidia/nemotron-3-super-120b-a12b",
#     "nvidia:z-ai/glm-5.3-flash",
# ]
# TIER1 = ["gemini:gemini-3.5-flash-lite", "groq:qwen/qwen3.8-27b", "nvidia:z-ai/glm-5.3-flash"]
# PLAN = [(model, "v1") for model in SCREEN_MODELS] + [(model, version) for version in ("v3b_med", "v3b_strict") for model in TIER1]
# SCREEN_RUNS = 2
# TIMEOUT, TRIES = 120, 2                       # a slow free endpoint fails in ~4 min instead of 15
# PAUSE = {"groq": 20, "gemini": 5, "nvidia": 3, "openrouter": 4, "ollama": 0}   # seconds between calls per free tier
# BY_QID = {record["qid"]: record for record in EVIDENCE}
# SCREEN_RECORDS = [BY_QID[qid] for qid in SCREEN.qid]

# def saved_keys():
#     if not ANSWERS_FILE.exists():
#         return set()
#     previous = pd.read_csv(ANSWERS_FILE)
#     return set(zip(previous.model, previous.version, previous.format, previous.run, previous.qid))

# def pending_jobs(model, version, done):
#     return [(run, record) for run in range(1, SCREEN_RUNS + 1) for record in SCREEN_RECORDS
#             if (model, version, "f0", run, record["qid"]) not in done]

# def probe(model):
#     """One quick 17 + 25 call (60 s, 1 try): is this model answering right now?"""
#     start = time.time()
#     try:
#         real_answer_check(model, timeout=60, tries=1)
#         return f"ok ({time.time() - start:.0f} s)"
#     except LLMError as error:
#         return f"FAIL - {str(error)[:150]}"

# def screen_model(model, version, jobs):
#     """Answers the pending screening questions with one model and prompt version; an error stops this pair only."""
#     pause = PAUSE[provider_of(model)[0]]
#     for number, (run, record) in enumerate(jobs, 1):
#         system, prompt = build(record, version, "f0")
#         try:
#             answer = chat(model, system, prompt, timeout=TIMEOUT, tries=TRIES)
#         except LLMError as error:
#             return f"stopped after {number - 1}/{len(jobs)} - {str(error)[:200]}"
#         pd.DataFrame([{"model": model, "version": version, "format": "f0", "run": run, "qid": record["qid"],
#                        "answer": answer}]).to_csv(ANSWERS_FILE, mode="a", header=not ANSWERS_FILE.exists(), index=False)
#         time.sleep(pause)
#     return f"done - {len(jobs)} answers"

# print(f"{len(SCREEN)} questions × {SCREEN_RUNS} runs = {len(SCREEN) * SCREEN_RUNS} answers per model and version")
# done = saved_keys()
# for model, version in PLAN:
#     jobs = pending_jobs(model, version, done)
#     if not jobs:
#         print(f"{model} | {version}: complete")
#         continue
#     check = probe(model)
#     if not check.startswith("ok"):
#         print(f"{model} | {version}: skipped - probe {check}")
#         continue
#     start = time.time()
#     print(f"{model} | {version}: {screen_model(model, version, jobs)} ({time.time() - start:.0f} s)", flush=True)

In [80]:
# 15.11a Screening setup - definitions only, no API calls
from src.llm import LLMError, provider_of

SCREEN_MODELS = [
    "gemini:gemini-3.5-flash-lite",
    "groq:openai/gpt-oss-120b",
    "groq:openai/gpt-oss-20b",
    "groq:qwen/qwen3.8-27b",
    "nvidia:google/gemma-4-31b-it",
    "nvidia:nvidia/nemotron-3-super-120b-a12b",
    "nvidia:z-ai/glm-5.3-flash",
]
TIER1 = ["gemini:gemini-3.5-flash-lite", "groq:qwen/qwen3.8-27b", "nvidia:z-ai/glm-5.3-flash"]
PLAN = [(model, "v1") for model in SCREEN_MODELS] + [(model, version) for version in ("v3b_med", "v3b_strict") for model in TIER1]
SCREEN_RUNS = 2
TIMEOUT, TRIES = 120, 2                       # a slow free endpoint fails in ~4 min instead of 15
PAUSE = {"groq": 20, "gemini": 5, "nvidia": 3, "openrouter": 4, "ollama": 0}   # seconds between calls per free tier
BY_QID = {record["qid"]: record for record in EVIDENCE}
SCREEN_RECORDS = [BY_QID[qid] for qid in SCREEN.qid]

def saved_keys():
    if not ANSWERS_FILE.exists():
        return set()
    previous = pd.read_csv(ANSWERS_FILE)
    return set(zip(previous.model, previous.version, previous.format, previous.run, previous.qid))

def pending_jobs(model, version, done):
    return [(run, record) for run in range(1, SCREEN_RUNS + 1) for record in SCREEN_RECORDS
            if (model, version, "f0", run, record["qid"]) not in done]

def probe(model):
    """One quick 17 + 25 call (60 s, 1 try): is this model answering right now?"""
    start = time.time()
    try:
        real_answer_check(model, timeout=60, tries=1)
        return f"ok ({time.time() - start:.0f} s)"
    except LLMError as error:
        return f"FAIL - {str(error)[:150]}"

def screen_model(model, version, jobs):
    """Answers the pending screening questions with one model and prompt version; an error stops this pair only."""
    pause = PAUSE[provider_of(model)[0]]
    for number, (run, record) in enumerate(jobs, 1):
        system, prompt = build(record, version, "f0")
        try:
            answer = chat(model, system, prompt, timeout=TIMEOUT, tries=TRIES)
        except LLMError as error:
            return f"stopped after {number - 1}/{len(jobs)} - {str(error)[:200]}"
        pd.DataFrame([{"model": model, "version": version, "format": "f0", "run": run, "qid": record["qid"],
                       "answer": answer}]).to_csv(ANSWERS_FILE, mode="a", header=not ANSWERS_FILE.exists(), index=False)
        time.sleep(pause)
    return f"done - {len(jobs)} answers"

In [81]:
# 15.11b Screening run - prints pending calls first; calls APIs only when ALLOW_API_CALLS is True
ALLOW_API_CALLS = False

done = saved_keys()
pending = {pair: pending_jobs(*pair, done) for pair in PLAN}
print(f"{len(SCREEN)} questions × {SCREEN_RUNS} runs = {len(SCREEN) * SCREEN_RUNS} answers per model and version")
print({f"{model} | {version}": len(jobs) for (model, version), jobs in pending.items() if jobs} or "Nothing pending")
print(f"Total pending calls: {sum(len(jobs) for jobs in pending.values())}\n")

if ALLOW_API_CALLS:
    for (model, version), jobs in pending.items():
        if not jobs:
            print(f"{model} | {version}: complete")
            continue
        check = probe(model)
        if not check.startswith("ok"):
            print(f"{model} | {version}: skipped - probe {check}")
            continue
        start = time.time()
        print(f"{model} | {version}: {screen_model(model, version, jobs)} ({time.time() - start:.0f} s)", flush=True)
else:
    print("ALLOW_API_CALLS is False - no API calls made.")

40 questions × 2 runs = 80 answers per model and version
Nothing pending
Total pending calls: 0

ALLOW_API_CALLS is False - no API calls made.


In [82]:
print(PLAN)

[('gemini:gemini-3.5-flash-lite', 'v1'), ('groq:openai/gpt-oss-120b', 'v1'), ('groq:openai/gpt-oss-20b', 'v1'), ('groq:qwen/qwen3.8-27b', 'v1'), ('nvidia:google/gemma-4-31b-it', 'v1'), ('nvidia:nvidia/nemotron-3-super-120b-a12b', 'v1'), ('nvidia:z-ai/glm-5.3-flash', 'v1'), ('gemini:gemini-3.5-flash-lite', 'v3b_med'), ('groq:qwen/qwen3.8-27b', 'v3b_med'), ('nvidia:z-ai/glm-5.3-flash', 'v3b_med'), ('gemini:gemini-3.5-flash-lite', 'v3b_strict'), ('groq:qwen/qwen3.8-27b', 'v3b_strict'), ('nvidia:z-ai/glm-5.3-flash', 'v3b_strict')]


In [83]:
print(list(VERSIONS.keys()))

['v1', 'v3b_med', 'v3b_strict']


In [84]:
# 15.12 Score the screening: one row per model and prompt version (f0, the 40 questions) - mean of the runs and their spread
import pandas as pd
def attach_evidence(answers):
    """Adds what the scorer needs: question data, the exact prompt that was sent and the source texts."""
    answers = answers.copy()
    for column in ("source", "label", "question", "action"):
        answers[column] = answers.qid.map(lambda qid: BY_QID[qid][column])
    answers["expected"] = answers.qid.map(lambda qid: BY_QID[qid]["expected"] or "")
    answers["prompt"] = [build(BY_QID[row.qid], row.version, row.format)[1] for row in answers.itertuples()]
    answers["sources"] = answers.qid.map(lambda qid: [hit["text"] for hit in BY_QID[qid]["hits"]])
    return answers

screen_answers = pd.read_csv(ANSWERS_FILE).fillna({"answer": ""})
planned = (screen_answers.model + " | " + screen_answers.version).isin({f"{model} | {version}" for model, version in PLAN})
screen_answers = screen_answers[planned & screen_answers.qid.isin(set(SCREEN.qid)) & screen_answers.format.eq("f0")
                                & screen_answers.run.le(SCREEN_RUNS)]
SCREEN_SCORED = ev.score_run(attach_evidence(screen_answers))
display(pd.crosstab([SCREEN_SCORED.model, SCREEN_SCORED.version], SCREEN_SCORED.error.replace("", "ok")))

error                                                improper citation  \
model                                    version                         
gemini:gemini-3.5-flash-lite             v1                          6   
                                         v3b_med                     7   
                                         v3b_strict                  7   
groq:openai/gpt-oss-120b                 v1                         16   
groq:openai/gpt-oss-20b                  v1                         15   
groq:qwen/qwen3.8-27b                    v1                          6   
                                         v3b_med                     4   
                                         v3b_strict                  2   
nvidia:google/gemma-4-31b-it             v1                          8   
nvidia:nvidia/nemotron-3-super-120b-a12b v1                         13   
nvidia:z-ai/glm-5.3-flash                v1                          6   
                                         v3b_med                     6   
                                         v3b_strict                  4   

error                                                inaccurate answer  ok  \
model                                    version                             
gemini:gemini-3.5-flash-lite             v1                          4  60   
                                         v3b_med                     4  64   
                                         v3b_strict                  4  57   
groq:openai/gpt-oss-120b                 v1                          6  52   
groq:openai/gpt-oss-20b                  v1                          5  52   
groq:qwen/qwen3.8-27b                    v1                          6  62   
                                         v3b_med                    10  62   
                                         v3b_strict                 12  56   
nvidia:google/gemma-4-31b-it             v1                          5  63   
nvidia:nvidia/nemotron-3-super-120b-a12b v1                         11  53   
nvidia:z-ai/glm-5.3-flash                v1                          3  67   
                                         v3b_med                     4  66   
                                         v3b_strict                  4  66   

error                                                over-responsiveness  \
model                                    version                           
gemini:gemini-3.5-flash-lite             v1                            2   
                                         v3b_med                       2   
                                         v3b_strict                    2   
groq:openai/gpt-oss-120b                 v1                            2   
groq:openai/gpt-oss-20b                  v1                            6   
groq:qwen/qwen3.8-27b                    v1                            2   
                                         v3b_med                       2   
                                         v3b_strict                    2   
nvidia:google/gemma-4-31b-it             v1                            4   
nvidia:nvidia/nemotron-3-super-120b-a12b v1                            2   
nvidia:z-ai/glm-5.3-flash                v1                            4   
                                         v3b_med                       4   
                                         v3b_strict                    4   

error                                                unwarranted refusal  
model                                    version                          
gemini:gemini-3.5-flash-lite             v1                            8  
                                         v3b_med                       3  
                                         v3b_strict                   10  
groq:openai/gpt-oss-120b                 v1                            4  
groq:openai/gpt-oss-20b                  v1                            2  
groq:qwen/qwen3.8-27b                    v1         

In [85]:
# 15.15 Scorecard - only the three numbers our goal depends on, per model and prompt version (complete pairs only)
def scorecard(scored):
    refused = scored.refused.astype(bool)
    answerable, unanswerable = scored.llm_label.eq("answerable"), scored.llm_label.eq("unanswerable")
    correct = ~refused & scored.correctness.astype(float).fillna(0).ge(ev.TAU)
    cited = scored.cite_recall.astype(float)[~refused].dropna()
    return pd.Series({"correct on answerable %": 100 * correct[answerable].mean(),
                      "over-answer on unanswerable %": 100 * (~refused)[unanswerable].mean(),
                      "statements cited %": 100 * cited.mean(),
                      "answers": len(scored)})

CARD = SCREEN_SCORED.groupby(["model", "version"]).apply(scorecard).round(1)
display(CARD[CARD.answers == SCREEN_RUNS * len(SCREEN)].sort_values(["model", "version"]))

C:\Users\amanc\AppData\Local\Temp\ipykernel_23652\640722627.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  CARD = SCREEN_SCORED.groupby(["model", "version"]).apply(scorecard).round(1)


correct on answerable %  \
model                                    version                               
gemini:gemini-3.5-flash-lite             v1                             78.6   
                                         v3b_med                        87.5   
                                         v3b_strict                     75.0   
groq:openai/gpt-oss-120b                 v1                             82.1   
groq:openai/gpt-oss-20b                  v1                             87.5   
groq:qwen/qwen3.8-27b                    v1                             82.1   
                                         v3b_med                        78.6   
                                         v3b_strict                     64.3   
nvidia:google/gemma-4-31b-it             v1                             91.1   
nvidia:nvidia/nemotron-3-super-120b-a12b v1                             78.6   
nvidia:z-ai/glm-5.3-flash                v1                             94.6   
                                         v3b_med                        92.9   
                                         v3b_strict                     89.3   

                                                     over-answer on unanswerable %  \
model                                    version                                     
gemini:gemini-3.5-flash-lite             v1                                    8.3   
                                         v3b_med                               8.3   
                                         v3b_strict                            8.3   
groq:openai/gpt-oss-120b                 v1                                    8.3   
groq:openai/gpt-oss-20b                  v1                                   25.0   
groq:qwen/qwen3.8-27b                    v1                                    8.3   
                                         v3b_med                               8.3   
                                         v3b_strict                            8.3   
nvidia:google/gemma-4-31b-it             v1                                   16.7   
nvidia:nvidia/nemotron-3-super-120b-a12b v1                                    8.3   
nvidia:z-ai/glm-5.3-flash                v1                                   16.7   
                                         v3b_med                              16.7   
                                         v3b_strict                           16.7   

                                                     statements cited %  \
model                                    version                          
gemini:gemini-3.5-flash-lite             v1                        80.9   
                                         v3b_med                   75.5   
                                         v3b_strict                75.7   
groq:openai/gpt-oss-120b                 v1                        63.9   
groq:openai/gpt-oss-20b                  v1                        63.0   
groq:qwen/qwen3.8-27b                    v1                        78.7   
                                         v3b_med                   80.6   
                                         v3b_strict                84.9   
nvidia:google/gemma-4-31b-it             v1                        76.2   
nvidia:nvidia/nemotron-3-super-120b-a12b v1                        66.7   
nvidia:z-ai/glm-5.3-flash                v1                        76.7   
                                         v3b_med                   73.1   
                                         v3b_strict                74.8   

                                                     answers  
model                                    version              
gemini:gemini-3.5-flash-lite             v1             80.0  
                                         v3b_med        80.0  
                                         v3b_strict     80.0  
groq:openai/gpt-oss-120b                 v1             80.0  
groq:openai/gpt-oss-20b                  v1             8

In [86]:
SCORECARD = getattr(scorecard, "data", scorecard)   # put the name of the table 15.15 displays; .data unwraps a styled table

E2_MODELS = ["gemini:gemini-3.5-flash-lite", "groq:qwen/qwen3.8-27b", "nvidia:z-ai/glm-5.3-flash"]
E2_VERSIONS = ["v1", "v3b_med", "v3b_strict"]
RETRIEVAL_MISS_QIDS = {"NHM09"}   # same for every model -> fixed via PDF page metadata, not by model choice
ACCEPTED_CORRECTIONS = set()      # official Option 1; set to {"C-055"} to see the sensitivity
P_ANSWERABLE = 0.7                # share of answerable questions (28 of 40 in the screen set)
K_VALUES = [1, 2, 3]
CHOSEN_K = 3                      # harm ratio: 1 over-answer = 3 missed answers (decide before E3)
SCORECARD = SCREEN_SCORED.groupby(["model", "version"]).apply(scorecard)
e2_rows = SCREEN_SCORED[SCREEN_SCORED["model"].isin(E2_MODELS) & SCREEN_SCORED["version"].isin(E2_VERSIONS)]
unanswerable_rows = e2_rows[(e2_rows["llm_label"] == "unanswerable") & ~e2_rows["qid"].isin(RETRIEVAL_MISS_QIDS)]
real_over_answer = ~unanswerable_rows["refused"] & ~unanswerable_rows["qid"].isin(ACCEPTED_CORRECTIONS)
over_rate = real_over_answer.groupby([unanswerable_rows["model"], unanswerable_rows["version"]]).mean()

trade_off = SCORECARD.loc[over_rate.index, ["correct on answerable %", "statements cited %"]].copy()
trade_off["real over-answer %"] = (100 * over_rate).round(1)
miss_cost = 100 * P_ANSWERABLE * (1 - trade_off["correct on answerable %"] / 100)
over_cost = 100 * (1 - P_ANSWERABLE) * over_rate
for k in K_VALUES:
    trade_off[f"cost k={k}"] = (miss_cost + over_cost * k).round(1)

trade_off = trade_off.sort_values([f"cost k={CHOSEN_K}", "statements cited %"], ascending=[True, False])
selected = trade_off.index[0]
trade_off["beats selected if k <"] = ((miss_cost[selected] - miss_cost) / (over_cost - over_cost[selected])).where(over_cost > over_cost[selected]).round(2)

print(trade_off.to_string())
print(f"\nSelected at k={CHOSEN_K}: {selected[0]} with prompt {selected[1]}")

                                         correct on answerable %  statements cited %  real over-answer %  cost k=1  cost k=2  cost k=3  beats selected if k <
model                        version                                                                                                                         
gemini:gemini-3.5-flash-lite v3b_med                   87.500000           75.528998                 0.0       8.8       8.8       8.8                    NaN
nvidia:z-ai/glm-5.3-flash    v1                        94.642857           76.704148                 9.1       6.5       9.2      11.9                   1.83
groq:qwen/qwen3.8-27b        v1                        82.142857           78.713200                 0.0      12.5      12.5      12.5                    NaN
nvidia:z-ai/glm-5.3-flash    v3b_med                   92.857143           73.065854                 9.1       7.7      10.5      13.2                   1.38
gemini:gemini-3.5-flash-lite v1                     

C:\Users\amanc\AppData\Local\Temp\ipykernel_23652\3387354766.py:10: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  SCORECARD = SCREEN_SCORED.groupby(["model", "version"]).apply(scorecard)


In [87]:
record = BY_QID["NHM09"]
print({key: value for key, value in record.items() if key != "hits"}, "\n")
for number, hit in enumerate(record["hits"], 1):
    print(f"[{number}] " + " | ".join(f"{key}={value}" for key, value in hit.items() if key != "text"))
    print(hit["text"][:300], "\n")

{'source': 'manual', 'qid': 'NHM09', 'label': 'answerable', 'question': 'can you tell for Appointment and Onboarding which page of pdf I should look', 'expected': "Chapter 6 'Appointment and Onboarding' is on printed pages 17-18 (PDF 37-38): 6.1 Joining formalities (offer letter, ID card, attendance), 6.2 Orientation, 6.3 Probation.", 'fixed': 'can you tell for Appointment and Onboarding which page of pdf I should look', 'action': 'answer', 'final': 'S', 'signals': {'best': 1.57, 'hard_out': [], 'soft_out': [], 'missing': []}} 

[1] chunk_id=S034-01 | doc_type=section | path=Chapter 6: Appointment and Onboarding > 6.1 Joining Formalities > 6.1.1 Appointment / Offer Letter | printed_pages=['17']
6.1.1. Appointment / Offer Letter On the day of joining, the new employees should get the original appointment letter (of the same offer letter that had been sent on email after selection). The appointment letter should then be signed by the employee and the employer (SHS/DHS). 

[2] chunk_id=S0

In [88]:
from src.paths import PAGES_OCR, LETTERS_JSONL, PAGE_MAP_CSV, PAGE_TYPES_CSV, PAGES_FINAL
PAGES_FILE = PAGES_FINAL # path to pages.jsonl from notebook 04

pages = pd.read_json(PAGES_FILE, lines=True)
numbered = pages.dropna(subset=["printed_page"]).copy()
numbered["printed_page"] = numbered["printed_page"].map(lambda value: str(int(value)) if isinstance(value, float) else str(value))
repeated = numbered[numbered.duplicated("printed_page", keep=False)]

print(numbered.set_index("printed_page").loc[["17", "18"], "pdf_page"], "\n")
print(f"{len(numbered)} pages have a printed number; {repeated['printed_page'].nunique()} printed numbers appear on more than one PDF page")
print(repeated[["printed_page", "pdf_page", "page_type", "part"]].to_string())

printed_page
17    37
18    38
Name: pdf_page, dtype: int64 

172 pages have a printed number; 0 printed numbers appear on more than one PDF page
Empty DataFrame
Columns: [printed_page, pdf_page, page_type, part]
Index: []


In [89]:
# C-002 check: removing the old statements() must not change any score (no API calls)
import importlib, inspect, subprocess, types
from src import evaluate as ev
from src.paths import ROOT

ev = importlib.reload(ev)
print("new parser loaded:", "CITE.sub" in inspect.getsource(ev.statements))

old_source = subprocess.run(["git", "show", "pre-refactor:src/evaluate.py"], cwd=ROOT,
                            capture_output=True, text=True, encoding="utf-8", check=True).stdout
ev_old = types.ModuleType("evaluate_old")
exec(old_source, ev_old.__dict__)

prepared = attach_evidence(screen_answers)
new_scored, old_scored = ev.score_run(prepared), ev_old.score_run(prepared)
print("all scores identical:", new_scored.equals(old_scored))
display(new_scored.groupby(["model", "version"]).apply(scorecard).round(1))

new parser loaded: True
all scores identical: True


C:\Users\amanc\AppData\Local\Temp\ipykernel_23652\1201647338.py:17: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  display(new_scored.groupby(["model", "version"]).apply(scorecard).round(1))


correct on answerable %  \
model                                    version                               
gemini:gemini-3.5-flash-lite             v1                             78.6   
                                         v3b_med                        87.5   
                                         v3b_strict                     75.0   
groq:openai/gpt-oss-120b                 v1                             82.1   
groq:openai/gpt-oss-20b                  v1                             87.5   
groq:qwen/qwen3.8-27b                    v1                             82.1   
                                         v3b_med                        78.6   
                                         v3b_strict                     64.3   
nvidia:google/gemma-4-31b-it             v1                             91.1   
nvidia:nvidia/nemotron-3-super-120b-a12b v1                             78.6   
nvidia:z-ai/glm-5.3-flash                v1                             94.6   
                                         v3b_med                        92.9   
                                         v3b_strict                     89.3   

                                                     over-answer on unanswerable %  \
model                                    version                                     
gemini:gemini-3.5-flash-lite             v1                                    8.3   
                                         v3b_med                               8.3   
                                         v3b_strict                            8.3   
groq:openai/gpt-oss-120b                 v1                                    8.3   
groq:openai/gpt-oss-20b                  v1                                   25.0   
groq:qwen/qwen3.8-27b                    v1                                    8.3   
                                         v3b_med                               8.3   
                                         v3b_strict                            8.3   
nvidia:google/gemma-4-31b-it             v1                                   16.7   
nvidia:nvidia/nemotron-3-super-120b-a12b v1                                    8.3   
nvidia:z-ai/glm-5.3-flash                v1                                   16.7   
                                         v3b_med                              16.7   
                                         v3b_strict                           16.7   

                                                     statements cited %  \
model                                    version                          
gemini:gemini-3.5-flash-lite             v1                        80.9   
                                         v3b_med                   75.5   
                                         v3b_strict                75.7   
groq:openai/gpt-oss-120b                 v1                        63.9   
groq:openai/gpt-oss-20b                  v1                        63.0   
groq:qwen/qwen3.8-27b                    v1                        78.7   
                                         v3b_med                   80.6   
                                         v3b_strict                84.9   
nvidia:google/gemma-4-31b-it             v1                        76.2   
nvidia:nvidia/nemotron-3-super-120b-a12b v1                        66.7   
nvidia:z-ai/glm-5.3-flash                v1                        76.7   
                                         v3b_med                   73.1   
                                         v3b_strict                74.8   

                                                     answers  
model                                    version              
gemini:gemini-3.5-flash-lite             v1             80.0  
                                         v3b_med        80.0  
                                         v3b_strict     80.0  
groq:openai/gpt-oss-120b                 v1             80.0  
groq:openai/gpt-oss-20b                  v1             8

In [90]:
# C-004 check: scoring from src/ (file in, no notebook state) must reproduce notebook 15's E2 results (no API calls)
import importlib
ev = importlib.reload(ev)

e2_rows = ev.load_scoring_input(EVAL / "e2_scoring_input.jsonl")
e2_screen = ev.select(e2_rows, PLAN, SCREEN.qid, fmt="f0", max_run=SCREEN_RUNS)
e2_scored = ev.score_run(e2_screen[ev.SCORING_COLUMNS])
print("scored rows:", len(e2_scored), "| same as notebook:",
      e2_scored[SCREEN_SCORED.columns].equals(SCREEN_SCORED.reset_index(drop=True)))

e2_cards = ev.scorecards(e2_scored)
print("scorecard same as notebook:", e2_cards.equals(SCREEN_SCORED.groupby(["model", "version"]).apply(scorecard)))

e2_table, e2_selected = ev.trade_off(e2_scored, E2_MODELS, E2_VERSIONS, RETRIEVAL_MISS_QIDS, ACCEPTED_CORRECTIONS,
                                     P_ANSWERABLE, K_VALUES, CHOSEN_K)
print("trade-off same as notebook:", e2_table.equals(trade_off), "| selected:", e2_selected)
print(e2_table.to_string())

scored rows: 1040 | same as notebook: True
scorecard same as notebook: True
trade-off same as notebook: True | selected: ('gemini:gemini-3.5-flash-lite', 'v3b_med')
                                         correct on answerable %  statements cited %  real over-answer %  cost k=1  cost k=2  cost k=3  beats selected if k <
model                        version                                                                                                                         
gemini:gemini-3.5-flash-lite v3b_med                   87.500000           75.528998                 0.0       8.8       8.8       8.8                    NaN
nvidia:z-ai/glm-5.3-flash    v1                        94.642857           76.704148                 9.1       6.5       9.2      11.9                   1.83
groq:qwen/qwen3.8-27b        v1                        82.142857           78.713200                 0.0      12.5      12.5      12.5                    NaN
nvidia:z-ai/glm-5.3-flash    v3b_med         

D:\SwasthyaSetu\src\evaluate.py:225: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return scored.groupby(["model", "version"]).apply(scorecard)
C:\Users\amanc\AppData\Local\Temp\ipykernel_23652\4025944677.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print("scorecard same as notebook:", e2_cards.equals(SCREEN_SCORED.groupby(["model", "version"]).apply(scorecard)))
D:\SwasthyaSetu\src\evaluate.py:225: FutureWar